In [8]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector


In [9]:
X = np.array([
    [0.004, 70,  0.02],
    [-0.003, 40, -0.01],
    [0.006, 80,  0.03],
    [-0.005, 25, -0.04]
])
y = np.array([1, 0, 1, 0])


In [10]:
def minmax(x, xmin, xmax):
    return (x - xmin) / (xmax - xmin)

Xn = np.zeros_like(X, dtype=float)
Xn[:,0] = minmax(X[:,0], -0.01, 0.01)   # return
Xn[:,1] = X[:,1] / 100.0               # RSI
Xn[:,2] = minmax(X[:,2], -0.05, 0.05)  # momentum



In [11]:
def qnn_circuit(x, params):
    qc = QuantumCircuit(3)
    qc.ry(np.pi * x[0], 0)
    qc.ry(np.pi * x[1], 1)
    qc.ry(np.pi * x[2], 2)
    qc.ry(params[0], 0)
    qc.ry(params[1], 1)
    qc.ry(params[2], 2)
    qc.cx(0, 1)
    qc.cx(1, 2)
    return qc



In [15]:

def predict_prob(x, params):
    qc = qnn_circuit(x, params)
    print(qc.draw())
    sv = Statevector.from_instruction(qc)
    
    print( sv )
    probs = sv.probabilities()
    return probs[1] + probs[3] + probs[5] + probs[7]  # qubit-0 = |1>



In [13]:
def loss(params):
    eps = 1e-9
    L = 0
    for i in range(len(Xn)):
        p = predict_prob(Xn[i], params)
        L += -(y[i]*np.log(p+eps) + (1-y[i])*np.log(1-p+eps))
    return L / len(Xn)



In [14]:
params = np.random.uniform(0, np.pi, 3)
lr = 0.3

for epoch in range(50):
    grads = np.zeros_like(params)
    for k in range(len(params)):
        shift = np.zeros_like(params)
        shift[k] = np.pi / 2
        grads[k] = (loss(params + shift) - loss(params - shift)) / 2
    params -= lr * grads
    if epoch % 10 == 0:
        print(f"epoch {epoch}, loss {loss(params):.4f}")

print("trained params:", params)








     ┌───────────┐ ┌────────────┐          
q_0: ┤ Ry(7π/10) ├─┤ Ry(2.6166) ├──■───────
     ├───────────┤ ├────────────┤┌─┴─┐     
q_1: ┤ Ry(7π/10) ├─┤ Ry(1.1754) ├┤ X ├──■──
     ├───────────┤┌┴────────────┤└───┘┌─┴─┐
q_2: ┤ Ry(7π/10) ├┤ Ry(0.90704) ├─────┤ X ├
     └───────────┘└─────────────┘     └───┘
Statevector:
 Statevector([ 0.00152909+0.j,  0.01178364+0.j, -0.73754071+0.j,
             -0.07780483+0.j,  0.08629259+0.j,  0.66499609+0.j,
             -0.01306912+0.j, -0.00137869+0.j],
            dims=(2, 2, 2))
Probabilities:
 [2.33812381e-06 1.38854105e-04 5.43966296e-01 6.05359229e-03
 7.44641043e-03 4.42219806e-01 1.70801833e-04 1.90078809e-06]
     ┌────────────┐ ┌────────────┐          
q_0: ┤ Ry(1.0996) ├─┤ Ry(2.6166) ├──■───────
     └┬──────────┬┘ ├────────────┤┌─┴─┐     
q_1: ─┤ Ry(2π/5) ├──┤ Ry(1.1754) ├┤ X ├──■──
      ├──────────┤ ┌┴────────────┤└───┘┌─┴─┐
q_2: ─┤ Ry(2π/5) ├─┤ Ry(0.90704) ├─────┤ X ├
      └──────────┘ └─────────────┘     └───┘
Statevector:
 Statev